# IntAct / MINT (IMEx) — Protein-Protein Interaction Data

**IntAct** is a freely available, open-source database and analysis system for protein interaction data, maintained by EMBL-EBI. **MINT** (Molecular INTeraction database) was originally developed at the University of Rome "Tor Vergata" and is now curated collaboratively through the **IMEx Consortium**.

The **IMEx Consortium** (International Molecular Exchange Consortium) is a group of major public interaction databases that share curation effort and apply a common, deep curation standard based on the **PSI-MI controlled vocabulary**. All IMEx records are assigned a globally unique **IMEx ID** and are exchanged in **PSI-MI XML 2.5** or **MITAB** format.

---

## Key Facts

| Property | Value |
|---|---|
| Full name | IntAct Molecular Interaction Database |
| Host institution | EMBL-EBI (Hinxton, UK) |
| IMEx partner | Yes — lead partner of the IMEx Consortium |
| Data licence | Creative Commons Attribution 4.0 (CC-BY 4.0) |
| Primary format | PSI-MI XML 2.5 / MITAB 2.5–2.7 |
| Interaction types | Physical, genetic, co-localization |
| REST API base | `https://www.ebi.ac.uk/intact/ws/interaction` |
| FTP bulk download | `https://ftp.ebi.ac.uk/pub/databases/intact/current/psimitab/` |
| Species coverage | All — >750,000 curated binary interactions (2024) |

---

## References

- Orchard S. *et al.* (2014). **The MIntAct project — IntAct as a common curation platform for 11 molecular interaction databases.** *Nucleic Acids Research*, 42(D1), D358–D363. https://doi.org/10.1093/nar/gkt1115
- IMEx Consortium: https://www.imexconsortium.org
- ELIXIR Core Data Resource: https://elixir-europe.org/platforms/data/core-data-resources

In [ ]:
import requests
import time
from pathlib import Path
import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to IntAct PSICQUIC REST API and confirm access
    * [x] Download clustered MITAB interaction file (`intact-micluster.zip`) from EBI FTP with caching
    * [x] Parse MITAB 2.7 columns into a Polars DataFrame with correct dtypes
    * [x] Save parsed data to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise DataFrame dimensions, null rates, duplicate interactions
    * [ ] Standardise identifier namespaces (UniProtKB, Ensembl, etc.)
    * [ ] Filter by detection method and interaction type
    * [ ] Map IMEx IDs to interaction records
* [ ] **Analysis**
    * [ ] Compute degree distribution of the interaction network
    * [ ] Identify hub proteins (high-degree nodes)
    * [ ] Compare evidence types (experimental vs. inferred)
* [ ] **Visualization**
    * [ ] Plot degree distribution (log-log scale)
    * [ ] Draw sub-network for a protein of interest using NetworkX
* [ ] **Statistical analysis**
    * [ ] Fit power-law distribution to degree sequence
    * [ ] Compute network clustering coefficient

## 1. Ingest Data

### 1.1 PSICQUIC API — Quick Connectivity Check

The **PSICQUIC** (Protemics Standard Initiative Common QUery Interface) REST API provides programmatic access to IntAct interaction data. We query it here as a lightweight connectivity test and to retrieve a small sample of interactions for the well-studied tumour suppressor **TP53** (UniProtKB: P04637).

In [ ]:
PSICQUIC_BASE = (
    "https://www.ebi.ac.uk/Tools/webservices/psicquic/intact"
    "/webservices/current/search"
)
FTP_BASE = "https://ftp.ebi.ac.uk/pub/databases/intact/current/psimitab"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# MITAB 2.7 column names (15 standard columns)
MITAB27_COLS = [
    "idA", "idB",
    "altA", "altB",
    "aliasA", "aliasB",
    "method",
    "author",
    "publication",
    "taxonA", "taxonB",
    "interaction_type",
    "source_db",
    "interaction_id",
    "confidence",
]

# ── Connectivity check: total interaction count ──────────────────────────────
count_url = f"{PSICQUIC_BASE}/query/*?format=count"
resp = requests.get(count_url, timeout=30)
resp.raise_for_status()
total_interactions = int(resp.text.strip())
print(f"IntAct PSICQUIC — total interactions available: {total_interactions:,}")

# ── Small test query: TP53 (P04637) interactions ─────────────────────────────
tp53_url = (
    f"{PSICQUIC_BASE}/query/P04637"
    "?format=tab27&firstResult=0&maxResults=100"
)
resp = requests.get(tp53_url, timeout=30)
resp.raise_for_status()

lines = [ln for ln in resp.text.strip().splitlines() if not ln.startswith("#")]
print(f"TP53 sample rows returned: {len(lines)}")

# Parse the sample into a small Polars DataFrame for inspection
tp53_df = pl.read_csv(
    resp.content,
    separator="\t",
    has_header=False,
    comment_prefix="#",
    new_columns=MITAB27_COLS,
    infer_schema_length=0,          # keep everything as Utf8 for now
)
print(tp53_df.shape)
tp53_df.head(5)

### 1.2 Download Clustered MITAB File

The **clustered** MITAB file (`intact-micluster.zip`, ~50 MB) merges redundant evidence for the same interacting pair into single rows with aggregated scores, making it a practical starting point for network analysis. The full file (`intact.zip`, ~500 MB) preserves individual evidence lines.

We download with caching: if the zip already exists in `data/` it is reused without a network request.

In [ ]:
ZIP_URL  = f"{FTP_BASE}/intact-micluster.zip"
ZIP_PATH = DATA_DIR / "intact-micluster.zip"

def download_with_progress(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    """Stream-download *url* to *dest*, printing MB progress."""
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(chunk_size=chunk_size):
                fh.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB  ({pct:.0f}%)",
                          end="", flush=True)
    print(f"\nSaved → {dest}")

if ZIP_PATH.exists():
    print(f"Cache hit — using existing {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Downloading {ZIP_URL} …")
    download_with_progress(ZIP_URL, ZIP_PATH)

ZIP_PATH.stat().st_size / 1e6  # MB

### 1.3 Parse MITAB into a Polars DataFrame

The zip contains a single tab-separated text file (`intact-micluster.txt`). We read it directly from the zip archive using Polars' built-in support for in-memory buffers, assign the 15 standard MITAB 2.7 column names, and cache the result as a Parquet file for fast subsequent loads.

In [ ]:
import zipfile, io

PARQUET_PATH = DATA_DIR / "intact_micluster.parquet"

def load_mitab_from_zip(zip_path: Path) -> pl.DataFrame:
    """Read the first .txt entry of *zip_path* into a Polars DataFrame."""
    with zipfile.ZipFile(zip_path) as zf:
        # Identify the data file (ignore any README / checksum entries)
        txt_names = [n for n in zf.namelist() if n.endswith(".txt")]
        if not txt_names:
            raise FileNotFoundError("No .txt file found inside the zip archive.")
        data_name = txt_names[0]
        print(f"Reading '{data_name}' from zip …")
        with zf.open(data_name) as fh:
            raw = fh.read()

    df = pl.read_csv(
        io.BytesIO(raw),
        separator="\t",
        has_header=True,           # clustered file includes a header line
        infer_schema_length=0,     # read everything as Utf8 initially
        truncate_ragged_lines=True,
    )

    # The header in the file uses #-prefixed names; rename to clean MITAB27_COLS
    df = df.rename({old: new for old, new in zip(df.columns, MITAB27_COLS)})
    return df

if PARQUET_PATH.exists():
    print(f"Cache hit — loading from {PARQUET_PATH}")
    interactions = pl.read_parquet(PARQUET_PATH)
else:
    interactions = load_mitab_from_zip(ZIP_PATH)
    interactions.write_parquet(PARQUET_PATH)
    print(f"Saved Parquet cache → {PARQUET_PATH}")

print(f"\nDataFrame shape : {interactions.shape}")
print(f"Columns         : {interactions.columns}")
interactions.head(3)

### 1.4 Extract UniProt Accessions

MITAB `idA` / `idB` values are formatted as `namespace:accession` (e.g. `uniprotkb:P04637`). We split on `:` to produce clean `accA` / `accB` columns containing bare accession numbers, and add a boolean flag indicating whether both interactors are UniProtKB entries — the majority of IntAct's curated pairs.

In [ ]:
def extract_accession(col: pl.Expr) -> pl.Expr:
    """Return the part after the first ':' in a MITAB identifier column.

    E.g.  'uniprotkb:P04637'  ->  'P04637'
          'intact:EBI-123456' ->  'EBI-123456'
    Rows that contain no ':' are returned unchanged.
    """
    return (
        col.str.splitn(":", 2)
           .list.get(1)
           .fill_null(col)   # fallback: keep original if no colon present
    )

interactions = interactions.with_columns(
    extract_accession(pl.col("idA")).alias("accA"),
    extract_accession(pl.col("idB")).alias("accB"),
    (
        pl.col("idA").str.starts_with("uniprotkb:") &
        pl.col("idB").str.starts_with("uniprotkb:")
    ).alias("both_uniprot"),
)

n_uniprot = interactions["both_uniprot"].sum()
pct = n_uniprot / len(interactions) * 100
print(f"Total interactions    : {len(interactions):,}")
print(f"Both interactors UniProtKB: {n_uniprot:,}  ({pct:.1f}%)")

# Sample of extracted accessions
interactions.select(["idA", "accA", "idB", "accB", "both_uniprot"]).head(6)